# PPI-only — Image-head ablations on Google Colab

Runs **Ablation C** — **ResNet50** & **ConvNeXt-Tiny** (pooling `max`/`avg`, ProteinLoss) — for the **bp, cc, mf** ontologies on a Colab GPU, because they are too slow on a local 4 GB card.

It reuses the exact `ppi_only` pipeline (batch 32, 200 epochs, patience 25, seed 1337) with **no methodology changes**, and writes per-run metrics in the **same format** as the local runs so results merge back cleanly.

**Flow:** pick GPU → set paths → mount Drive → stage code+data locally → verify (sanity + smoke) → run the 12 image runs → bundle results back to Drive.

> The MLP/CNN1D vector heads (Ablations A & B) are run locally; only the image heads run here.

## 1. Pick a GPU
**Runtime → Change runtime type → T4 GPU** (or better), then run the cell below.

In [ ]:
!nvidia-smi

## 2. Configure paths  ✏️ **EDIT THESE**
Upload your `projeto` folder (with `ppi_only/`, `ppi_v4/ppi_v4/`, `configs/`) and your data folder to Google Drive, then point the variables below at them.

In [ ]:
# === EDIT to match your Google Drive layout ===
# Folder containing ppi_only/, ppi_v4/, configs/
PROJECT_SRC = "/content/drive/MyDrive/projeto-ppi-only/projeto"
# Folder containing ppi.csv, go.obo, {bp,cc,mf}_{train,val,test,ic}.csv
DATA_SRC    = "/content/drive/MyDrive/projeto-ppi-only/data"
# Where to write the results bundle back on Drive
RESULTS_OUT = "/content/drive/MyDrive/projeto-ppi-only/colab_results"

# Colab GPU runtimes usually have 2 vCPUs. num_workers only affects SPEED, not results
# (features are deterministic; the shuffle order is seeded in the main process).
NUM_WORKERS = 2

LOCAL = "/content/projeto"   # fast local working copy (don't edit)

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Stage code + data onto local disk
Copies the code and data to Colab's fast local disk (Drive I/O is slow for 200-epoch training).

In [ ]:
import os, shutil
if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
shutil.copytree(PROJECT_SRC, LOCAL, symlinks=False,
                ignore=shutil.ignore_patterns('runs', 'results', 'data', '.ipynb_checkpoints', '__pycache__'))
os.makedirs(f"{LOCAL}/data", exist_ok=True)
for f in os.listdir(DATA_SRC):
    if f.endswith(('.csv', '.obo')):
        shutil.copy(f"{DATA_SRC}/{f}", f"{LOCAL}/data/{f}")

assert os.path.exists(f"{LOCAL}/ppi_only/run_ablations.py"), "ppi_only/ not found under PROJECT_SRC"
assert os.path.exists(f"{LOCAL}/ppi_v4/ppi_v4/engine.py"),   "ppi_v4/ppi_v4/ not found under PROJECT_SRC"
need = {'ppi.csv', 'go.obo'} | {f"{d}_{s}.csv" for d in ('bp','cc','mf') for s in ('train','val','test','ic')}
have = set(os.listdir(f"{LOCAL}/data"))
missing = need - have
assert not missing, f"data files missing: {sorted(missing)}"
print("OK — staged at", LOCAL)
print("data:", sorted(have))

## 5. Dependencies
Colab ships PyTorch + torchvision (CUDA). This just fills any gaps.

In [ ]:
import importlib.util, subprocess, sys
pip_name = {'yaml': 'pyyaml', 'sklearn': 'scikit-learn'}
need = [pip_name.get(p, p) for p in ('yaml','matplotlib','pandas','numpy','tqdm','sklearn')
        if importlib.util.find_spec(p) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 6. Set num_workers (results-neutral speedup)

In [ ]:
import re, pathlib
base = pathlib.Path(LOCAL) / "configs/ppi_only/base.yaml"
lines = base.read_text().splitlines()
lines = [re.sub(r'^num_workers:.*', f'num_workers: {NUM_WORKERS}        # Colab', ln)
         if ln.strip().startswith('num_workers:') else ln for ln in lines]
base.write_text("\n".join(lines) + "\n")
print("num_workers ->", NUM_WORKERS)

## 7. Verify — sanity gate + 1-epoch image smoke
If the sanity gate fails (exit code 2) or a smoke run errors, STOP and check paths/data before the full matrix.

In [ ]:
%cd /content/projeto
!python -m ppi_only.main --sanity-only --domain bp --model resnet50

In [ ]:
# 1 epoch on a small subset for both archs — should finish in ~1 min each on a T4
!python -m ppi_only.main --config configs/ppi_only/mock_image.yaml --domain bp --arch resnet50      --smoke
!python -m ppi_only.main --config configs/ppi_only/mock_image.yaml --domain bp --arch convnext_tiny --smoke

## 8. Run the image ablations (Ablation C) — bp, cc, mf
ResNet50 & ConvNeXt-Tiny, pooling max/avg → **12 runs** (2 archs × 2 poolings × 3 domains).
Cost-ordered; per-run failures are logged and don't abort the matrix.

⚠️ **Colab session limits:** free sessions disconnect after ~12 h or on idle. The two archs are split into separate cells so you can **bundle results (Step 9) after each** as a checkpoint. If a session drops, re-run Steps 3–6 then continue.

In [ ]:
%cd /content/projeto
# ResNet50: bp,cc,mf × {max,avg} = 6 runs
!python -u -m ppi_only.run_ablations --domains bp,cc,mf --only-model resnet50

In [ ]:
%cd /content/projeto
# ConvNeXt-Tiny: bp,cc,mf × {max,avg} = 6 runs
!python -u -m ppi_only.run_ablations --domains bp,cc,mf --only-model convnext_tiny

## 9. Bundle results back to Drive
Run this after each arch (checkpoint) and/or at the end. Collects per-run metrics, manifests, weights, and curves.

In [ ]:
import os, glob, shutil, time
os.makedirs(RESULTS_OUT, exist_ok=True)
stamp = time.strftime("%Y%m%d_%H%M%S")
bundle = f"/content/ppi_image_results_{stamp}"
for pat in ["results/*_metrics.csv", "results/*_metrics.json", "results/*_manifest.json",
            "results/ABLATIONS_summary.*", "results/_ablation_status.json",
            "runs/*_best.pt", "runs/*_losses.png", "runs/*_loss_hist.json"]:
    for f in glob.glob(f"/content/projeto/{pat}"):
        dst = os.path.join(bundle, os.path.relpath(f, "/content/projeto"))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
zip_path = shutil.make_archive(f"{RESULTS_OUT}/ppi_image_results_{stamp}", "zip", bundle)
n = len(glob.glob(f"{bundle}/results/*_metrics.csv"))
print(f"Saved bundle to Drive: {zip_path}")
print(f"Per-run metric files in bundle: {n}  (expect up to 12 when both archs are done)")

## 10. Back on your local machine
1. Download `ppi_image_results_*.zip` from `RESULTS_OUT` on your Drive.
2. Unzip its `results/*` and `runs/*` into your local `projeto/results/` and `projeto/runs/`.
3. Tell Claude Code **"the image results are in"** — it will regenerate the full combined
   `results/ABLATIONS_summary.{csv,md}` + `results/RUN_REPORT.md` covering **A + B + C** and report best **wFmax(test)** per ablation/domain.

**Reproducibility note:** metrics are identical whether `num_workers` is 0 or 2 — features are deterministic and the DataLoader shuffle is seeded in the main process; `num_workers` changes only speed.